In [0]:
dbutils.widgets.text("day_folder", "retail_day_03")
day_folder = dbutils.widgets.get("day_folder")

In [0]:
dbutils.widgets.text("day_folder", "retail_day_03")
day_folder = dbutils.widgets.get("day_folder")

from pyspark.sql import functions as F

schema = "workspace.retail_lakehouse"
landing = f"/Volumes/workspace/retail_lakehouse/landing/{day_folder}"

for name in ["customers", "products", "orders"]:
    df = (spark.read
          .option("header", "true").option("inferSchema", "true")
          .csv(f"{landing}/{name}.csv")
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source_file", F.lit(f"{name}.csv")))
    (df.write.format("delta").mode("append")
       .saveAsTable(f"{schema}.bronze_{name}"))
    print(f"appended {df.count()} rows to bronze_{name} from {day_folder}")